# Better Stress Experiments for `MDSTorusProjector`

This notebook is a bounded experiment harness for stress minimization variants that could replace or inform `sgd_minibatch_njit` inside `MDSTorusProjector.fit_transform(...)`.

Scope:
- reuse the existing graph families from `comparison.ipynb`
- keep runtime bounded and memory usage conservative
- compare candidate optimizers on the same graphs, seeds, and pair budget
- rank candidates by final stress and runtime


In [ ]:
import warnings
import pandas as pd
%load_ext autoreload
%autoreload 2

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", message="Unable to import Axes3D")

from modules import stress_experiments as experiments


## Imports / Setup

The benchmark uses a fixed pair budget per run to keep comparisons bounded. All kernels are warmed up once before timing so the runtime measurements are not dominated by Numba compilation.


In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

BENCHMARK_KWARGS = {
    "learning_rate": 1.0,
    "max_iters": 4000,
    "batch_pairs": 4096,
    "alpha_init": 1.0,
    "alpha_ema": 0.05,
    "learn_mode": "alpha",
    "geom_lr": 0.01,
    "theta": np.pi / 2,
}
SEEDS = (0, 1, 2)
CHEN_INDICES = None
VARIANT_NAMES = [variant.name for variant in experiments.VARIANTS]
VARIANT_NAMES.remove("full_sweep_sync_sqrtnorm")
VARIANT_NAMES.remove("anchor_unique_online")
VARIANT_NAMES.remove("coverage_unique_sync_sqrtnorm")
VARIANT_NAMES.remove("sampled_unique_sync_batchavg")
VARIANT_NAMES.remove("sampled_unique_sync_countnorm")
VARIANT_NAMES.remove("sampled_unique_sync_sqrtnorm")
VARIANT_NAMES.remove("sampled_unique_sync_hybridnorm")
VARIANT_NAMES.remove("stratified_distance_sync_sqrtnorm")
print("Variants:", VARIANT_NAMES, "")

print("Warming up Numba kernels...")
experiments.warmup_variants()
print("Warmup complete.")


## Reusable Experimental Kernels

These candidates keep a compatible high-level signature while changing the optimization mechanics:

- `baseline_random_minibatch`: current behavior
- `sampled_unique_online`: unique unordered pairs, online updates
- `sampled_unique_sync_batchavg`: unique unordered pairs, synchronous batch update
- `sampled_unique_sync_countnorm`: synchronous batch update with per-node averaging
- `sampled_unique_sync_sqrtnorm`: synchronous batch update with inverse-sqrt visit normalization
- `sampled_unique_sync_hybridnorm`: early `sqrtnorm`, late `countnorm`
- `stratified_distance_sync_sqrtnorm`: `sqrtnorm` with short/medium/long distance-balanced minibatches
- `coverage_unique_sync_sqrtnorm`: `sqrtnorm` with coverage-constrained node-round minibatches
- `anchor_unique_online`: anchor sweep with a few unique partners per node
- `full_sweep_sync_sqrtnorm`: small-graph full sweep control


In [ ]:
variant_catalog = experiments.variant_catalog_df()
display(variant_catalog)


## Evaluation Helpers

The benchmark harness records:
- runtime
- normalized stress under the fitted rectangular torus geometry
- distortion
- `SGS` and neighborhood precision for graphs up to 400 nodes


In [ ]:
def top_variants_for_dataset(dataset_name: str, dataset_means: pd.DataFrame, k: int = 3) -> list[str]:
    rows = (
        dataset_means.loc[dataset_means["dataset"] == dataset_name]
        .sort_values(["mean_stress", "mean_runtime_s"])
        .head(k)
    )
    return rows["variant"].tolist()


def display_summary_tables(results_df: pd.DataFrame):
    summary = experiments.summarize_results(results_df)
    recommendation_df = experiments.recommendation_table(results_df)

    display(Markdown("### Overall mean/median performance"))
    display(summary["overall"])

    display(Markdown("### Mean performance by graph family"))
    display(summary["by_group"])

    display(Markdown("### Best mean variant per dataset"))
    display(summary["best_by_dataset"])

    display(Markdown("### Recommendation ranking"))
    display(recommendation_df)

    return summary, recommendation_df


## Benchmark Graph Construction / Loading

The dataset families match `comparison.ipynb`:
- periodic grids
- Chen graphs from `chengraphs/*.json`
- planted partition graphs

The Chen set is intentionally a representative subset here to keep the notebook bounded.


In [ ]:
benchmark_specs = experiments.build_representative_benchmarks(
    chen_indices=CHEN_INDICES,
    include_large_grid=False,
    planted_seed=0,
)
benchmark_catalog = experiments.benchmark_catalog_df(benchmark_specs)
display(benchmark_catalog)


## Benchmark Execution

Every variant is run on the same graph, the same seeds, and the same nominal pair budget (`max_iters * batch_pairs`). The full-sweep control converts that budget into a bounded number of full passes and skips larger graphs explicitly.


In [ ]:
results_df, embedding_store = experiments.benchmark_variants(
    benchmark_specs,
    variant_names=VARIANT_NAMES,
    seeds=SEEDS,
    compute_extra_metrics=True,
    max_extra_metric_n=400,
    store_embeddings=True,
    **BENCHMARK_KWARGS,
)

display(results_df.head(20))
print("Completed runs:", int((results_df["status"] == "ok").sum()))
print("Skipped runs:", int((results_df["status"] == "skipped").sum()))


In [ ]:
ok_results = results_df.loc[results_df["status"] == "ok"].copy()
summary, recommendation_df = display_summary_tables(results_df)
dataset_means = summary["dataset_means"]


In [ ]:
overall_plot = summary["overall"].copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(overall_plot["mean_runtime_s"], overall_plot["mean_stress"], s=70)
for _, row in overall_plot.iterrows():
    ax.text(row["mean_runtime_s"], row["mean_stress"], row["variant"], fontsize=8)
ax.set_xlabel("Mean runtime [s]")
ax.set_ylabel("Mean stress")
ax.set_title("Runtime / stress tradeoff")
ax.grid(alpha=0.2)
plt.show()


## Controlled Test 1: Same Optimizers Under Euclidean Stress

This control keeps the optimizer mechanics fixed but swaps the torus objective for plain Euclidean pairwise stress. If the same variant still wins, the effect is mostly optimizer-side rather than torus-specific.


In [ ]:
control_specs = [
    spec for spec in benchmark_specs
    if spec.name in {"grid_20x20", "planted_k4_n40", "planted_k4_n80"} or spec.name.startswith("chen_")
]
# control_variant_names = [
#     "baseline_random_minibatch",
#     "sampled_unique_online",
#     "sampled_unique_sync_countnorm",
#     "sampled_unique_sync_sqrtnorm",
# ]
control_variant_names = VARIANT_NAMES

objective_control_df = experiments.benchmark_objective_controls(
    control_specs,
    variant_names=control_variant_names,
    seeds=SEEDS,
    learning_rate=BENCHMARK_KWARGS["learning_rate"],
    max_iters=BENCHMARK_KWARGS["max_iters"],
    batch_pairs=BENCHMARK_KWARGS["batch_pairs"],
    alpha_init=BENCHMARK_KWARGS["alpha_init"],
    alpha_ema=BENCHMARK_KWARGS["alpha_ema"],
    theta=BENCHMARK_KWARGS["theta"],
)
objective_summary = experiments.summarize_objective_controls(objective_control_df)

display(Markdown("### Overall by objective"))
display(objective_summary["overall"])

display(Markdown("### By graph family and objective"))
display(objective_summary["by_group"])


## Controlled Test 2: Torus Visitation and Movement Diagnostics

This diagnostic keeps the torus objective fixed and logs per-epoch statistics for visit imbalance, pair-gradient magnitudes, and resulting node movement magnitudes.


In [ ]:
diagnostic_datasets = ["grid_20x20", "chen_10", "planted_k4_n80"]
# diagnostic_variants = [
#     "baseline_random_minibatch",
#     "sampled_unique_sync_countnorm",
#     "sampled_unique_sync_sqrtnorm",
# ]
diagnostic_variants = VARIANT_NAMES
# diagnostic_variants.remove("sampled_unique_online")

diagnostic_rows = []
for dataset_name in diagnostic_datasets:
    spec = next(spec for spec in benchmark_specs if spec.name == dataset_name)
    for variant_name in diagnostic_variants:
        diag_df = experiments.diagnose_torus_optimization(
            spec.distances,
            variant_name,
            learning_rate=BENCHMARK_KWARGS["learning_rate"],
            max_iters=BENCHMARK_KWARGS["max_iters"],
            batch_pairs=BENCHMARK_KWARGS["batch_pairs"],
            seed=0,
            alpha_init=BENCHMARK_KWARGS["alpha_init"],
            alpha_ema=BENCHMARK_KWARGS["alpha_ema"],
            theta=BENCHMARK_KWARGS["theta"],
        )
        diag_df["dataset"] = dataset_name
        diagnostic_rows.append(diag_df)

diagnostic_df = pd.concat(diagnostic_rows, ignore_index=True)
diagnostic_summary = (
    diagnostic_df.groupby(["dataset", "variant"], as_index=False)
    .agg(
        mean_visit_cv=("visit_cv", "mean"),
        mean_pair_grad_norm=("mean_pair_grad_norm", "mean"),
        mean_node_move_norm=("mean_node_move_norm", "mean"),
        max_node_move_norm=("max_node_move_norm", "mean"),
    )
    .sort_values(["dataset", "mean_visit_cv", "mean_node_move_norm"])
)
display(diagnostic_summary)


In [ ]:
fig, axes = plt.subplots(len(diagnostic_datasets), 2, figsize=(10, 3.2 * len(diagnostic_datasets)), sharex=True)
if len(diagnostic_datasets) == 1:
    axes = np.array([axes])

for row_idx, dataset_name in enumerate(diagnostic_datasets):
    subset = diagnostic_df.loc[diagnostic_df["dataset"] == dataset_name]
    for variant_name in diagnostic_variants:
        variant_subset = subset.loc[subset["variant"] == variant_name]
        axes[row_idx, 0].plot(variant_subset["epoch"], variant_subset["visit_cv"], label=variant_name)
        axes[row_idx, 1].plot(variant_subset["epoch"], variant_subset["mean_node_move_norm"], label=variant_name)
    axes[row_idx, 0].set_title(f"{dataset_name}: visit CV")
    axes[row_idx, 1].set_title(f"{dataset_name}: mean node move norm")
    axes[row_idx, 0].grid(alpha=0.2)
    axes[row_idx, 1].grid(alpha=0.2)

axes[0, 0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()


## Broader Scenario Suite

This repeats the control experiments on a wider variety of graph families:

- torus-native periodic graphs
- Euclidean-native planar and geometric graphs
- random / structured generator graphs
- Chen graphs
- planted partition graphs

For the torus objective, the original learning-rate schedule is kept. For the Euclidean control, a more conservative schedule is used so the comparison reflects optimizer behavior rather than numerical blow-up.


In [ ]:
diverse_specs = experiments.build_diverse_control_benchmarks(seed=0)
# diverse_variant_names = [
#     "baseline_random_minibatch",
#     "sampled_unique_online",
#     "sampled_unique_sync_countnorm",
#     "sampled_unique_sync_sqrtnorm",
#     "sampled_unique_sync_hybridnorm",
#     "stratified_distance_sync_sqrtnorm",
#     "coverage_unique_sync_sqrtnorm",
# ]
diverse_variant_names = VARIANT_NAMES
# diverse_variant_names.remove("sampled_unique_sync_batchavg")


diverse_seeds = (0, 1)
display(experiments.benchmark_catalog_df(diverse_specs))


In [ ]:
diverse_torus_df = experiments.benchmark_objective_controls(
    diverse_specs,
    variant_names=diverse_variant_names,
    objectives=("torus",),
    seeds=diverse_seeds,
    learning_rate=1.0,
    max_iters=4000,
    batch_pairs=2048,
    alpha_init=1.0,
    alpha_ema=0.05,
    theta=np.pi / 2,
)
diverse_torus_df = diverse_torus_df.loc[diverse_torus_df["objective"] == "torus"].copy()
diverse_torus_summary = experiments.summarize_objective_controls(diverse_torus_df)

display(Markdown("### Broader torus summary"))
display(diverse_torus_summary["overall"])
display(diverse_torus_summary["by_group"])


In [ ]:
diverse_euclidean_df = experiments.benchmark_objective_controls(
    diverse_specs,
    variant_names=diverse_variant_names,
    objectives=("euclidean",),
    seeds=diverse_seeds,
    learning_rate=5.0,
    max_iters=120,
    batch_pairs=2048,
    alpha_init=1.0,
    alpha_ema=0.05,
    theta=np.pi / 2,
)
diverse_euclidean_df = diverse_euclidean_df.loc[diverse_euclidean_df["objective"] == "euclidean"].copy()
diverse_euclidean_summary = experiments.summarize_objective_controls(diverse_euclidean_df)

display(Markdown("### Broader Euclidean summary"))
display(diverse_euclidean_summary["overall"])
display(diverse_euclidean_summary["by_group"])


## Representative Embeddings

The plots below visualize the top variants for one representative graph from each family. This is not part of the ranking logic; it is only a qualitative check that the lower-stress candidates still produce coherent toroidal layouts.


In [ ]:
representative_datasets = [
    "grid_10x10",
    "grid_20x20",
    "grid_10x30",
    "chen_1",
    "chen_10",
    "chen_20",
    "planted_k4_n40",
    "planted_k4_n80",
]

for dataset_name in representative_datasets:
    spec = next(spec for spec in benchmark_specs if spec.name == dataset_name)
    chosen_variants = top_variants_for_dataset(dataset_name, dataset_means, k=3)
    print(dataset_name, "->", chosen_variants)
    experiments.plot_representative_embeddings(
        spec,
        embedding_store,
        chosen_variants,
        seed=0,
    )
    plt.show()


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from modules import graphio, metrics, stress_experiments
from modules.projector import MDSTorusProjector


def regular_grid_init(nx, ny):
    graph = graphio.periodic_lattice_graph(nx, ny)
    nodes = list(graph.nodes())
    pos = np.asarray([(i / nx, j / ny) for i, j in nodes], dtype=np.float64)
    return pos, nodes, graph


def _wrapped_delta(p, q):
    d = q - p
    return (d + 0.5) % 1.0 - 0.5


def run_grid_layout(
    nx,
    ny,
    *,
    learn_mode="alpha",
    fixed_aspect=None,
    max_iters=10000,
    batch_size=4096,
    seed=0,
    projection="wrap",
):
    """
    Run one grid-graph layout experiment.

    Cases:
    - learn_mode="rectangular"      -> current rectangular joint learner
    - learn_mode="alpha_aspect"     -> constrained alpha+aspect learner
    - fixed_aspect=2.0               -> force aspect 2.0 and optimize coordinates with fixed radii
    """
    graph, distances = graphio.get_periodic_lattice(nx, ny)
    nodes = list(graph.nodes())

    proj = MDSTorusProjector(projection=projection)

    fit_kwargs = dict(
        max_iters=max_iters,
        batch_size=batch_size,
        seed=seed,
    )

    if fixed_aspect is not None:
        aspect = float(fixed_aspect)
        if aspect <= 0.0:
            raise ValueError("fixed_aspect must be positive")
        aspect = max(aspect, 1.0 / aspect)
        fit_kwargs["learn_mode"] = "alpha"
        fit_kwargs["r0_init"] = 1.0 / np.sqrt(aspect)
        fit_kwargs["r1_init"] = np.sqrt(aspect)
    else:
        fit_kwargs["learn_mode"] = learn_mode

    X = proj.fit_transform(distances, **fit_kwargs)

    geod = stress_experiments.make_rect_torus_geod(
        proj.alpha_, proj.r0_, proj.r1_, proj.theta_
    )
    stress = metrics.geodesic_stress(X, distances, geod)
    aspect = max(proj.r0_, proj.r1_) / min(proj.r0_, proj.r1_)

    return {
        "nx": nx,
        "ny": ny,
        "graph": graph,
        "nodes": nodes,
        "distances": distances,
        "X": np.asarray(X, dtype=np.float64),
        "stress": float(stress),
        "alpha": float(proj.alpha_),
        "r0": float(proj.r0_),
        "r1": float(proj.r1_),
        "aspect": float(aspect),
        "learn_mode": fit_kwargs["learn_mode"],
        "fixed_aspect": fixed_aspect,
        "max_iters": max_iters,
        "batch_size": batch_size,
        "seed": seed,
        "projection": projection,
    }


def plot_grid_layout(
    result,
    *,
    ax=None,
    color_by="x",               # "x", "y", or "index"
    draw_edges=True,
    node_size=18,
    edge_alpha=0.12,
    edge_width=0.6,
    tile=1,                     # 1 = central torus only, 3 = show 3x3 tiling
    title=None,
):
    """
    Plot a wrapped torus layout.

    tile=3 is useful when you want to see how wrap-around edges behave.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    X = result["X"]
    graph = result["graph"]
    nodes = result["nodes"]
    nx = result["nx"]
    ny = result["ny"]

    if color_by == "x":
        c = np.asarray([i for i, j in nodes], dtype=float)
    elif color_by == "y":
        c = np.asarray([j for i, j in nodes], dtype=float)
    else:
        c = np.arange(len(nodes), dtype=float)

    shifts = [(0, 0)] if tile == 1 else [
        (sx, sy) for sx in (-1, 0, 1) for sy in (-1, 0, 1)
    ]

    if draw_edges:
        node_to_idx = {node: k for k, node in enumerate(nodes)}
        for u, v in graph.edges():
            i = node_to_idx[u]
            j = node_to_idx[v]
            p = X[i]
            q = X[j]
            d = _wrapped_delta(p, q)
            for sx, sy in shifts:
                p0 = p + np.array([sx, sy], dtype=float)
                p1 = p0 + d
                ax.plot(
                    [p0[0], p1[0]],
                    [p0[1], p1[1]],
                    color="black",
                    lw=edge_width,
                    alpha=edge_alpha,
                    zorder=1,
                )

    for sx, sy in shifts:
        XX = X + np.array([sx, sy], dtype=float)
        ax.scatter(
            XX[:, 0],
            XX[:, 1],
            c=c,
            cmap="viridis",
            s=node_size,
            linewidths=0,
            zorder=2,
        )

    if tile == 1:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    else:
        ax.set_xlim(-1, 2)
        ax.set_ylim(-1, 2)

    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    if title is None:
        if result["fixed_aspect"] is not None:
            mode_text = f"fixed aspect={result['fixed_aspect']}"
        else:
            mode_text = result["learn_mode"]
        title = (
            f"{result['nx']}x{result['ny']} | {mode_text}\n"
            f"stress={result['stress']:.4f}, alpha={result['alpha']:.3f}, aspect={result['aspect']:.3f}"
        )
    ax.set_title(title, fontsize=10)
    return ax


def compare_grid_layouts(results, *, ncols=3, figsize_per_ax=5, **plot_kwargs):
    n = len(results)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_ax * ncols, figsize_per_ax * nrows),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, result in zip(axes, results):
        plot_grid_layout(result, ax=ax, **plot_kwargs)

    for ax in axes[len(results):]:
        ax.axis("off")

    fig.tight_layout()
    return fig, axes

In [ ]:
# Compare learned modes on 10x30
res_rect = run_grid_layout(10, 30, learn_mode="rectangular")
res_mode4 = run_grid_layout(10, 30, learn_mode="alpha_aspect")
res_fix2  = run_grid_layout(10, 30, fixed_aspect=2.0)
res_fix3  = run_grid_layout(10, 30, fixed_aspect=3.0)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix2, res_fix3],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

In [ ]:
# Compare learned modes on 10x30
res_rect = run_grid_layout(20, 40, learn_mode="rectangular")
res_mode4 = run_grid_layout(20, 40, learn_mode="alpha_aspect")
res_fix2  = run_grid_layout(20, 40, fixed_aspect=1.5)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix2],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

In [ ]:
# Compare learned modes on 10x30
res_rect = run_grid_layout(15, 45, learn_mode="rectangular")
res_mode4 = run_grid_layout(15, 45, learn_mode="alpha_aspect")
res_fix2  = run_grid_layout(15, 45, fixed_aspect=2.0)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix2],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

In [ ]:
# Compare learned modes
res_rect = run_grid_layout(30, 60, learn_mode="rectangular")
res_mode4 = run_grid_layout(30, 60, learn_mode="alpha_aspect")
res_fix2  = run_grid_layout(30, 60, fixed_aspect=1.5)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix2],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

In [ ]:
# Scan-like comparison by hand on 20x60
results = [
    run_grid_layout(20, 60, fixed_aspect=a)
    for a in (1.0, 2.0, 3.0, 4.0, 6.0)
]
compare_grid_layouts(results, color_by="y", draw_edges=True, tile=1, ncols=3)
plt.show()

In [ ]:
# Show wrap structure in 3x3 tiling
res = run_grid_layout(10, 60, fixed_aspect=6.0)
plot_grid_layout(res, color_by="x", draw_edges=True, tile=3)
plt.show()

In [ ]:
# Compare learned modes
res_rect = run_grid_layout(5, 60, learn_mode="rectangular")
res_mode4 = run_grid_layout(5, 60, learn_mode="alpha_aspect")
res_fix1dot2  = run_grid_layout(5, 60, fixed_aspect=1.2)
res_fix1dot5  = run_grid_layout(5, 60, fixed_aspect=1.5)
res_fix2  = run_grid_layout(5, 60, fixed_aspect=2)
res_fix3  = run_grid_layout(5, 60, fixed_aspect=3)
res_fix4  = run_grid_layout(5, 60, fixed_aspect=4)
res_fix5  = run_grid_layout(5, 60, fixed_aspect=5)
res_fix6  = run_grid_layout(5, 60, fixed_aspect=6)
res_fix7  = run_grid_layout(5, 60, fixed_aspect=7)
res_fix8  = run_grid_layout(5, 60, fixed_aspect=8)
res_fix85  = run_grid_layout(5, 60, fixed_aspect=8.5)
res_fix9  = run_grid_layout(5, 60, fixed_aspect=9)
res_fix95  = run_grid_layout(5, 60, fixed_aspect=9.5)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix1dot2, res_fix1dot5, res_fix2, res_fix3, res_fix4, res_fix5, res_fix6, res_fix7, res_fix8, res_fix85, res_fix9, res_fix95],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

In [ ]:
# Compare learned modes
res_rect = run_grid_layout(10, 60, learn_mode="rectangular")
res_mode4 = run_grid_layout(10, 60, learn_mode="alpha_aspect")
res_fix15  = run_grid_layout(10, 60, fixed_aspect=1.5)
res_fix2  = run_grid_layout(10, 60, fixed_aspect=2)
res_fix3  = run_grid_layout(10, 60, fixed_aspect=3)
res_fix4  = run_grid_layout(10, 60, fixed_aspect=4)

compare_grid_layouts(
    [res_rect, res_mode4, res_fix15, res_fix2, res_fix3, res_fix4],
    color_by="x",
    draw_edges=True,
    tile=1,
)
plt.show()

## 14. Historical Fixed-Aspect Comparisons

These notebook cells still compare learned layouts against manually chosen fixed aspects, but the projector no longer exposes the earlier `aspect_scan` API.

Current notebook behavior:

- `learn_mode="rectangular"` runs the direct joint radii learner
- `learn_mode="alpha_aspect"` runs the constrained alpha+aspect learner
- `fixed_aspect=a` now means: keep the side-length ratio fixed at `a` and optimize coordinates with `learn_mode="alpha"`

This preserves the intended visual comparisons without depending on the removed `candidate_aspects` / `aspect_scan` projector interface.
